# Odhad soustavy rezidenční poptávky po energii pomocí zdánlivě nesouvisejících regresí

## Manažerské shrnutí

Regionální energetická společnost odhaduje dvourovnicovou **soustavu rezidenční poptávky po energii** — rozpočtové podíly elektřiny a zemního plynu jako funkce vlastní ceny, křížové ceny a příjmu domácnosti — pomocí **PROC SYSLIN** metodou zdánlivě nesouvisejících regresí (SUR). Obě rovnice podílů sdílejí společný rozpočet domácnosti, takže jejich chyby jsou křížově korelované; SUR odhaduje rovnice společně, aby tuto korelaci využil, a obnovuje znaménkově koherentní vlastní a křížové cenové efekty a dodává mezirovnicovou kovarianci a testy restrikcí, které analytik poptávky potřebuje.

## Zdroje dat

| Datová sada | Řádky | Zrno | Klíčové proměnné | Popis |
|---------|------|-------|---------------|-------------|
| `energy` | 60 | jeden řádek na měsíční tržní pozorování | `month`, `lp_elec`, `lp_gas`, `lincome`, `w_elec`, `w_gas` | Syntetický měsíční panel rezidenčního trhu s energií. `lp_elec`/`lp_gas` jsou logaritmy reálných cen elektřiny a zemního plynu; `lincome` je logaritmus reálného příjmu domácnosti; `w_elec`/`w_gas` jsou rozpočtové výdajové podíly na elektřinu a zemní plyn, generované ze známé poptávkové struktury typu AIDS plus korelovaného mezirovnicového šumu. |

# Odhad soustavy rezidenční poptávky po energii pomocí zdánlivě nesouvisejících regresí

Regionální plynárenská a elektrárenská společnost chce pochopit, jak domácnosti přerozdělují svůj energetický rozpočet mezi **elektřinu** a **zemní plyn** při změnách relativních cen a příjmu. Přirozeným rámcem je **soustava poptávky**: sada rovnic rozpočtových podílů odhadovaných společně.

Společný odhad je tím správným nástrojem ze dvou důvodů:

- Rovnice podílů čerpají ze společného rozpočtu domácnosti, takže jejich chyby jsou **křížově korelované** — když domácnost utratí více za elektřinu, utratí méně za plyn. Odhad rovnic společně pomocí **zdánlivě nesouvisejících regresí (SUR)** tuto korelaci využívá pro efektivitu.
- Ekonomická teorie ukládá **lineární restrikce** (sčítání do jedné, homogenita, symetrie), které systémový estimátor dokáže vynutit přímo.

`PROC SYSLIN` s volbou `SUR` je vytvořen přesně pro tuto situaci. Fituje každou rovnici podílu, odhadne mezirovnicovou kovarianci chyb a znovu nafituje systém s touto kovariancí, aby dodal efektivní, teoreticky koherentní elasticity — spolu s maticí kovariance mezi modely a společnými testy restrikcí.

V tomto notebooku:

1. Generujeme syntetický měsíční panel trhu s energií ze známé struktury podílů.
2. Fitujeme dvourovnicový systém podílů pomocí SUR.
3. Porovnáme společný fit SUR s odhadem rovnice po rovnici pomocí OLS.
4. Uložíme restrikci homogenity a přečteme její společný F-test.
5. Extrahujeme tabulku koeficientů pro reportování elasticit.

## Krok 1 — Generování syntetického panelu trhu s energií

Simulujeme 60 měsíčních pozorování malé **dvoustatkové soustavy poptávky po energii** (elektřina a zemní plyn). Proces generující data je linearizovaný model rozpočtových podílů typu AIDS:

```
w_elec = a1 + g11*lp_elec + g12*lp_gas + b1*lincome + e1
w_gas  = a2 + g21*lp_elec + g22*lp_gas + b2*lincome + e2
```

Záměrně vestavíme **mezirovnicovou korelaci**: když domácnosti utratí více za elektřinu, utratí méně za plyn, takže `e1` a `e2` jsou záporně korelované. Společný cenový faktor trhu s energií také žene oba logaritmy cen společně. To jsou rysy, které odmění společný odhad SUR oproti fitování každé rovnice zvlášť.

In [1]:
data energy;
   CALL streaminit(70123);

   /* True structural coefficients (linearized AIDS share system) */
   a1  = 0.55;  g11 =  0.12; g12 = -0.08; b1 = -0.030;
   a2  = 0.45;  g21 = -0.08; g22 =  0.10; b2 = -0.025;

   OPAKUJ month = 1 TO 60;
      /* A common energy-market price factor drives BOTH prices,
         creating the collinearity that makes the problem ill-posed. */
      energy_factor = rand('NORMAL', 0, 0.15);

      lp_elec = 4.6 + energy_factor + rand('NORMAL', 0, 0.05);
      lp_gas  = 4.2 + energy_factor + rand('NORMAL', 0, 0.05);
      lincome = 10.8 + 0.004*month + rand('NORMAL', 0, 0.06);

      /* Negatively correlated cross-equation errors (budget tradeoff) */
      shock = rand('NORMAL', 0, 0.010);
      e1 =  shock + rand('NORMAL', 0, 0.006);
      e2 = -shock + rand('NORMAL', 0, 0.006);

      w_elec = a1 + g11*lp_elec + g12*lp_gas + b1*lincome + e1;
      w_gas  = a2 + g21*lp_elec + g22*lp_gas + b2*lincome + e2;

      VÝSTUP;
   KONEC;

   PONECHAT month lp_elec lp_gas lincome w_elec w_gas;
SPUSTIT;

PROCEDURA PRŮMĚRY data=energy n mean std MIN MAX maxdec=3;
   ŠTÍTEK w_elec="Podíl elektřiny" w_gas="Podíl plynu"
          lp_elec="Log ceny elektřiny" lp_gas="Log ceny plynu" lincome="Log příjmu";
   PROMĚNNÁ w_elec w_gas lp_elec lp_gas lincome;
SPUSTIT;

                                                  The MEANS Procedure

 Variable  Label                      N           Mean     Std Dev     Minimum     Maximum
 -----------------------------------------------------------------------------------------
 w_elec    Podíl elektřiny           60          0.440       0.011       0.417       0.464
 w_gas     Podíl plynu               60          0.228       0.014       0.198       0.256
 lp_elec   Log ceny elektřiny        60          4.617       0.142       4.354       4.911
 lp_gas    Log ceny plynu            60          4.211       0.162       3.818       4.566
 lincome   Log příjmu                60         10.916       0.088      10.723      11.174
 -----------------------------------------------------------------------------------------




NOTE: DATA energy


NOTE: Wrote energy (60 rows, 6 columns).
NOTE: DATA elapsed:
  wall  0.02 seconds
  cpu   0.02 seconds
NOTE: PROC MEANS
NOTE: PROC MEANS statement used.


## Krok 2 — Základní odhad soustavy poptávky pomocí SUR

Obě rovnice podílů odhadujeme společně. Volba `SUR` říká `PROC SYSLIN`, aby odhadl mezirovnicovou kovarianci chyb a použil ji pro efektivní fit metodou feasible-GLS. Dva pojmenované příkazy `MODEL` (`elec` a `gas`) definují systém; každý regresuje rozpočtový podíl na dva logaritmy cen a logaritmus příjmu.

SYSLIN reportuje odhady parametrů každé rovnice a na konci **matici kovariance mezi modely** — odhadnutou kovarianci chyb obou rovnic, kterou SUR využívá.

In [2]:
PROCEDURA syslin data=energy sur;
   elec: MODEL w_elec = lp_elec lp_gas lincome;
   gas:  MODEL w_gas  = lp_elec lp_gas lincome;
SPUSTIT;


                       The SYSLIN Procedure

                  SUR Estimation

  Model elec Dependent Variable: w_elec

  Number of Observations                       60
  SSE                                      0.0048
  MSE                                    0.000085
  R-Square                                 0.3865
  Adj R-Sq                                 0.3537

                      Parameter Estimates

                                   Standard
  Variable     DF     Estimate        Error    t Value    Pr > |t|
  ----------  ---  ------------  ----------  ---------  ----------
  Intercept      1      0.801017    0.150270       5.33      0.0000
  LP_ELEC        1      0.112463    0.021244       5.29      0.0000
  LP_GAS         1     -0.097938    0.018646      -5.25      0.0000
  LINCOME        1     -0.042842    0.013238      -3.24      0.0020

  Model gas Dependent Variable: w_gas

  Number of Observations                       60
  SSE                                      0.


NOTE: PROC SYSLIN data=energy

NOTE: Using Python for simultaneous equations estimation (SUR)
NOTE: PROC SYSLIN completed.


## Krok 3 — Porovnání s odhadem rovnice po rovnici pomocí OLS

Abychom viděli, co nám SUR přináší, znovu nafitujeme tytéž dvě rovnice metodou nejmenších čtverců (výchozí metoda, jedna rovnice po druhé). OLS ignoruje mezirovnicovou korelaci chyb, takže je konzistentní, ale méně efektivní než SUR, když je tato korelace nenulová — jak je zde z konstrukce.

Porovnání obou tabulek koeficientů ukazuje, jak se odhady a jejich směrodatné chyby posunou, jakmile se zohlední struktura systému.

In [3]:
PROCEDURA syslin data=energy ols;
   elec: MODEL w_elec = lp_elec lp_gas lincome;
   gas:  MODEL w_gas  = lp_elec lp_gas lincome;
SPUSTIT;


                       The SYSLIN Procedure

                  OLS Estimation

  Model elec Dependent Variable: w_elec

  Number of Observations                       60
  SSE                                      0.0048
  MSE                                    0.000085
  R-Square                                 0.3865
  Adj R-Sq                                 0.3537

                      Parameter Estimates

                                   Standard
  Variable     DF     Estimate        Error    t Value    Pr > |t|
  ----------  ---  ------------  ----------  ---------  ----------
  Intercept      1      0.801017    0.155544       5.15      0.0000
  LP_ELEC        1      0.112463    0.021989       5.11      0.0000
  LP_GAS         1     -0.097938    0.019300      -5.07      0.0000
  LINCOME        1     -0.042842    0.013702      -3.13      0.0028

  Model gas Dependent Variable: w_gas

  Number of Observations                       60
  SSE                                      0.


NOTE: PROC SYSLIN data=energy

NOTE: Using Python for simultaneous equations estimation (OLS)
NOTE: PROC SYSLIN completed.


## Krok 4 — Uložení ekonomické teorie a její test

Teorie poptávky říká, že nominální cenové/příjmové efekty by měly splňovat **homogenitu nultého stupně**: škálování všech cen a příjmu ponechá reálnou poptávku nezměněnou, takže cenové a příjmové koeficienty uvnitř rovnice se sečtou na nulu. Toto uložíme na podíl elektřiny příkazem `RESTRICT`.

SYSLIN znovu nafituje systém s ohledem na omezení a reportuje F-test **Restriction Results** o tom, zda je restrikce v souladu s daty — přímý, společný test hypotézy homogenity.

In [4]:
PROCEDURA syslin data=energy sur;
   elec: MODEL w_elec = lp_elec lp_gas lincome;
   gas:  MODEL w_gas  = lp_elec lp_gas lincome;

   /* Homogeneity on the electricity share equation:
      price and income coefficients sum to zero. */
   restrict lp_elec + lp_gas + lincome = 0;
SPUSTIT;


                       The SYSLIN Procedure

                  SUR Estimation

  Model elec Dependent Variable: w_elec

  Number of Observations                       60
  SSE                                      0.0048
  MSE                                    0.000085
  R-Square                                 0.3865
  Adj R-Sq                                 0.3537

                      Parameter Estimates

                                   Standard
  Variable     DF     Estimate        Error    t Value    Pr > |t|
  ----------  ---  ------------  ----------  ---------  ----------
  Intercept      1      0.801017    0.150270       5.33      0.0000
  LP_ELEC        1      0.112463    0.021244       5.29      0.0000
  LP_GAS         1     -0.097938    0.018646      -5.25      0.0000
  LINCOME        1     -0.042842    0.013238      -3.24      0.0020

  Model gas Dependent Variable: w_gas

  Number of Observations                       60
  SSE                                      0.


NOTE: PROC SYSLIN data=energy

NOTE: Using Python for simultaneous equations estimation (SUR)
NOTE: PROC SYSLIN completed.


## Krok 5 — Zachycení odhadů pro reportování elasticit

Pro závěrečný report zachováme zkonvergované koeficienty pomocí `OUTEST=`. Výsledná datová sada nese jeden řádek na rovnici s odhady interceptu a sklonů plus statistikami shody, které vstupují do výpočtů vlastní a křížové cenové elasticity společnosti při podílech na výběrovém průměru. Pro záznam ji vytiskneme.

In [5]:
PROCEDURA syslin data=energy sur outest=demand_est;
   elec: MODEL w_elec = lp_elec lp_gas lincome;
   gas:  MODEL w_gas  = lp_elec lp_gas lincome;
SPUSTIT;

PROCEDURA TISK data=demand_est noobs;
   NÁZEV "Odhady koeficientů systému poptávky (SUR)";
SPUSTIT;
NÁZEV;


                       The SYSLIN Procedure

                  SUR Estimation

  Model elec Dependent Variable: w_elec

  Number of Observations                       60
  SSE                                      0.0048
  MSE                                    0.000085
  R-Square                                 0.3865
  Adj R-Sq                                 0.3537

                      Parameter Estimates

                                   Standard
  Variable     DF     Estimate        Error    t Value    Pr > |t|
  ----------  ---  ------------  ----------  ---------  ----------
  Intercept      1      0.801017    0.150270       5.33      0.0000
  LP_ELEC        1      0.112463    0.021244       5.29      0.0000
  LP_GAS         1     -0.097938    0.018646      -5.25      0.0000
  LINCOME        1     -0.042842    0.013238      -3.24      0.0020

  Model gas Dependent Variable: w_gas

  Number of Observations                       60
  SSE                                      0.


NOTE: PROC SYSLIN data=energy

NOTE: Using Python for simultaneous equations estimation (SUR)
NOTE: Wrote OUTEST= dataset demand_est (2 rows).
NOTE: PROC SYSLIN completed.
NOTE: PROC PRINT data=demand_est

NOTE: PROC PRINT completed: 2 observations printed, 12 variables


## Interpretace výsledků

**Znaménková koherence.** Odhady SUR obnoví substituční strukturu vestavěnou do dat. V rovnici podílu elektřiny je **vlastní koeficient log-ceny kladný** (`LP_ELEC` = 0.112), zatímco **křížový cenový koeficient je záporný** (`LP_GAS` = -0.098); v rovnici podílu plynu se vzorec zrcadlí (vlastní `LP_GAS` = 0.114, křížový `LP_ELEC` = -0.075). Každý vlastní i křížový cenový efekt je statisticky významný (každé `Pr > |t|` pod 0.005), takže substituční znaménka jsou pevně identifikována, nikoli artefakty zašuměného fitu.

**SUR využívá mezirovnicovou korelaci.** **Matice kovariance mezi modely** ukazuje záporný mimodiagonální prvek (-0.000068): domácnosti vyměňují výdaje za elektřinu za výdaje za plyn, přesně jak bylo zkonstruováno. Protože je tato kovariance nenulová, je společný odhad SUR efektivnější než fitování každé rovnice zvlášť — směrodatné chyby SUR v kroku 2 jsou rovnoměrně mírně menší než směrodatné chyby OLS rovnice po rovnici v kroku 3 (například směrodatná chyba plynového `LP_ELEC` klesá z 0.0264 při OLS na 0.0255 při SUR), zatímco bodové odhady se nemění. Toto těsnější odvození je odměnou za modelování systému místo dvou samostatných regresí.

**Teorie obstojí.** Uložení **homogenity nultého stupně** na podíl elektřiny (cenové a příjmové koeficienty sečtené na nulu) pomocí `RESTRICT` dalo F-test Restriction Results 2.14 s `Pr > F` = 0.149. Restrikce **není zamítnuta**, takže predikce homogenity z teorie spotřebitelské poptávky je v souladu s tímto vzorkem — společnost může s důvěrou přenést omezené, teoreticky koherentní odhady do reportování.

**Provozní využití.** Tabulka `OUTEST=` zachovává jeden řádek na rovnici s odhady interceptu a sklonů a statistikami shody. Tyto koeficienty se přímo převádějí na vlastní a křížové cenové elasticity a příjmovou (výdajovou) elasticitu při podílech na výběrovém průměru (`W_ELEC` ~ 0.44, `W_GAS` ~ 0.23 z kroku 1), což společnosti dává obhajitelný, teoreticky konzistentní základ pro prognózování poptávky a scénáře návrhu sazeb.